# 06 — Logfire Observability

## Objectives

- Read an agent-run trace: spans, tool calls, token & cost badges
- Label agents with `name=` and add your own spans with `logfire.span`
- Trigger a `ModelRetry` on purpose and *find it in the UI*
- Use the Agents view and LLMs view to compare behavior over time

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

## Your two lines, recap

```python
logfire.configure(send_to_logfire='if-token-present')
logfire.instrument_pydantic_ai()
```

Every run of every instrumented agent becomes one **trace**. If you haven't yet, run
`uv run logfire auth` in a terminal now (browser login; free tier is 10M spans/month) and open
the **Live view** at [logfire.pydantic.dev](https://logfire.pydantic.dev) next to this notebook —
this notebook is best read side-by-side with the UI.

Two additions make traces *yours*:

- `Agent(..., name='...')` — names the run span (otherwise it's inferred)
- `logfire.span('...')` — your own spans, nested wherever you open them

In [ ]:
from pydantic_ai import Agent, RunContext

inventory_agent = Agent(
    MODEL,
    name="inventory_agent",  # ← the trace will carry this name
    instructions="You manage a tiny inventory. Use your tools; never invent stock numbers.",
)

STOCK = {"ABC-1": 42, "GIZMO-9": 7, "WIDGET-3": 0}


@inventory_agent.tool
async def lookup_stock(ctx: RunContext[None], sku: str) -> str:
    """Look up current stock for a SKU."""
    with logfire.span("lookup_stock", sku=sku):  # ← your own span, inside the tool call
        quantity = STOCK.get(sku.upper())
    return "unknown SKU" if quantity is None else f"{quantity} in stock"


r = await inventory_agent.run("How many ABC-1 and WIDGET-3 do we have? What is the total?")
print(r.output)

## 🔭 What to look for in the trace

1. One trace = one `agent run`, named `inventory_agent`.
2. Inside it: a span per **model call**, each with the coin badge (model, tokens, USD).
3. Between model calls: spans for **tool calls** — `lookup_stock` — and *nested inside those*,
   your custom `lookup_stock` span with its `sku` attribute.
4. Click any span: arguments, results, timing, the full conversation rendered transcript-style.

> ✅ **Checkpoint** — you found your custom span with `sku='ABC-1'` in the UI. That's
> OpenTelemetry under the hood: the same spans could go to any OTLP backend — Logfire is
> just the polished Pydantic-flavored one.

## Now make a bug — and watch Logfire catch it

A flaky tool that fails *once* on purpose, then succeeds. The retry loop you met in
notebooks 03–04, this time visible from above:

In [ ]:
from pydantic_ai import ModelRetry

flaky_calls = {"count": 0}


@inventory_agent.tool
async def fetch_price(ctx: RunContext[None], sku: str) -> str:
    """Fetch the current price of a SKU. (Flaky on its first call per kernel.)"""
    flaky_calls["count"] += 1
    if flaky_calls["count"] == 1:
        raise ModelRetry("price service hiccup — call me again")
    return "$9.99"


r = await inventory_agent.run("What does ABC-1 cost?")
print(r.output)
print("fetch_price was called", flaky_calls["count"], "times")

> ✅ **Checkpoint** — the answer includes $9.99 and the counter says 2 calls.
>
> 🔭 **The hunt**: in the Logfire trace, find the *first* `fetch_price` attempt — it ends in
> an error span carrying your `ModelRetry` message, followed by the model trying again and the
> successful second call. In production this is *the* debugging move: the trace shows you which
> tool choked, with what arguments, without a single print statement.

## The two dashboards you'll actually use

- **Agents view** — per named agent: run counts, turns per run, tool-calling turns, exceptions,
  and cost per agent. Your `name='inventory_agent'` is what populates it.
- **LLMs view** — per model: latency, tokens/sec, error rates, cost coverage. This is where the
  notebook-02 model shoot-out lands automatically.

(No Logfire account? Everything above still ran. `send_to_logfire='if-token-present'` means
observability is a login away, not a code change.)

## 🏋️ Exercise — the joker and the critic

Create **two** named agents — `joker` (tells one-line jokes) and `critic` (rates any joke from
0–10, structured output optional). Give one of `joker`'s tools a custom `logfire.span`. Run each
agent **3×** with different prompts. Then in the Logfire **Agents view**: compare run counts,
cost per agent, and find your custom span inside one of joker's traces.

Your take: [`../solutions/06_two_agents_spans.ipynb`](../solutions/06_two_agents_spans.ipynb).

## Next

→ **[07_history_and_streaming.ipynb](07_history_and_streaming.ipynb)** — conversations that
survive between runs, and output that appears as it's generated.